[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/youssefHosni/prompt-engineering-book-2e/blob/main/notebooks/ch04_room_to_think.ipynb)

# Chapter 4 — Giving the Model Room to Think

The headline finding in this chapter's first draft was an artifact of the
author's own prompt. "Reply with only the number" suppressed the working and
produced a 2/12 baseline that had nothing to do with reasoning.

Both conditions are run here, because the difference between them *is* the
chapter.

In [ ]:
# Colab only: fetch the shared helpers and install the SDKs.
# Running locally? Skip this cell -- you already have them.
import os, sys
if "google.colab" in sys.modules:
    !pip -q install anthropic openai python-dotenv
    !git clone -q https://github.com/youssefHosni/prompt-engineering-book-2e /content/peb-book
    sys.path.insert(0, "/content/peb-book")
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")


In [ ]:
import re

from peb import ask

RUNS = 12

## §4.2 Four numbered steps in one pass

In [ ]:
EMAIL = """Hi - the Nordic 400 I ordered last month has started making a loud
grinding noise whenever I pull a shot, and the crema has gone. It was fine for
three weeks. Is this something I can fix or do I need to send it back?"""

steps = """For the customer email below, do the following in order:

1. Identify the product the customer is writing about.
2. Decide whether the message is a complaint, a question, or praise.
3. If it is a complaint, name the specific fault.
4. Draft a two-sentence reply.

Return each step on its own line, labelled.

Email: ```{email}```"""

text, _ = ask(steps.format(email=EMAIL), max_tokens=4000)
print(text.strip()[:700])
print(f"\nsteps present: {sum(text.count(f'{i}.') > 0 for i in range(1, 5))}/4")

## §4.4 The instruction that suppresses the working

One variable changes between these two conditions: whether the model is
allowed to show its steps. Measure the prompt you wrote, not the one you
meant to write.

In [ ]:
WORD_PROBLEM = ("A cafe buys 3 boxes of 12 cups and 2 boxes of 18 cups. "
                "It breaks 7 cups. How many cups does it have?")   # 36 + 36 - 7 = 65

for label, suffix in (("room to think", " Show your working, then give the answer."),
                      ("answer only",   " Reply with only the number.")):
    right = 0
    for _ in range(RUNS):
        text, _ = ask(WORD_PROBLEM + suffix, max_tokens=4000)
        digits = [int(s) for s in re.findall(r"\b\d+\b", text)]
        right += bool(digits) and digits[-1] == 65
    print(f"{label:<15} {right}/{RUNS} correct")